# Feature Analysis — Mel-Spectrograms & Audio Features

This notebook analyzes the audio features used by the model:
1. Mel-spectrogram comparisons across emotion classes
2. Energy and zero-crossing rate patterns
3. Average spectrogram per emotion (emotion fingerprints)
4. CNN feature map visualization (post-training)


In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display

from src.data.preprocessing import process_audio

plt.style.use('seaborn-v0_8-whitegrid')

CSV_PATH = '../data/processed/iemocap_6class.csv'
df = pd.read_csv(CSV_PATH)
EMOTIONS = ['ang', 'hap', 'neu', 'sad', 'fru', 'exc']
SR = 16000
N_MELS = 128
HOP = 256

print(f'Loaded {len(df)} utterances')

## 1. Side-by-Side Mel-Spectrograms (same speaker, different emotions)

In [ ]:
# Pick one sample per emotion — same session for fair comparison
samples = {emo: df[df['emotion'] == emo].sample(1, random_state=7).iloc[0] for emo in EMOTIONS}

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, emo in zip(axes.flat, EMOTIONS):
    row = samples[emo]
    y, sr = librosa.load(row['audio_path'], sr=SR)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, hop_length=HOP)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    img = librosa.display.specshow(log_mel, sr=sr, hop_length=HOP, x_axis='time', y_axis='mel',
                                   ax=ax, cmap='magma')
    ax.set_title(f'{emo.upper()}  |  "{row["text"][:35]}..."', fontsize=10, fontweight='bold')
    ax.set_xlabel('Time (s)')

plt.suptitle('Mel-Spectrogram Comparison Across Emotion Classes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../logs/feature_spectrograms.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Average Spectrogram per Emotion (Emotion Fingerprints)

Compute the mean Mel-spectrogram across all utterances of each class. This reveals distinctive spectro-temporal patterns per emotion.

In [ ]:
N_FRAMES = 500  # 8s max
avg_mels = {}

for emo in EMOTIONS:
    subset = df[df['emotion'] == emo].sample(min(50, len(df[df['emotion']==emo])), random_state=42)
    mels = []
    for _, row in subset.iterrows():
        try:
            y, sr = librosa.load(row['audio_path'], sr=SR)
            mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, hop_length=HOP)
            log_mel = librosa.power_to_db(mel, ref=np.max)
            # Pad/truncate to fixed length
            T = log_mel.shape[1]
            if T >= N_FRAMES:
                log_mel = log_mel[:, :N_FRAMES]
            else:
                log_mel = np.pad(log_mel, ((0,0),(0, N_FRAMES-T)))
            mels.append(log_mel)
        except Exception:
            continue
    if mels:
        avg_mels[emo] = np.mean(mels, axis=0)
    print(f'{emo}: averaged {len(mels)} spectrograms')

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, emo in zip(axes.flat, EMOTIONS):
    if emo not in avg_mels:
        continue
    times = librosa.frames_to_time(np.arange(N_FRAMES), sr=SR, hop_length=HOP)
    freqs = librosa.mel_frequencies(n_mels=N_MELS)
    im = ax.imshow(avg_mels[emo], aspect='auto', origin='lower', cmap='magma',
                   extent=[times[0], times[-1], freqs[0], freqs[-1]])
    ax.set_title(f'Average: {emo.upper()}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Frequency (Hz)')

plt.suptitle('Mean Mel-Spectrogram per Emotion Class (Emotion Fingerprints)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../logs/feature_avg_spectrograms.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Energy & Zero-Crossing Rate by Emotion

High-level acoustic features that correlate strongly with emotion category.

In [ ]:
records = []
for emo in EMOTIONS:
    subset = df[df['emotion'] == emo].sample(min(30, len(df[df['emotion']==emo])), random_state=0)
    for _, row in subset.iterrows():
        try:
            y, sr = librosa.load(row['audio_path'], sr=SR)
            rms = float(librosa.feature.rms(y=y).mean())
            zcr = float(librosa.feature.zero_crossing_rate(y).mean())
            tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
            records.append({'emotion': emo, 'rms_energy': rms, 'zcr': zcr, 'tempo': float(tempo)})
        except Exception:
            continue

feat_df = pd.DataFrame(records)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, feat, title in zip(axes, ['rms_energy', 'zcr', 'tempo'],
                           ['RMS Energy', 'Zero-Crossing Rate', 'Tempo (BPM)']):
    feat_df.boxplot(column=feat, by='emotion', ax=ax, patch_artist=True)
    ax.set_title(f'{title} by Emotion', fontsize=12)
    ax.set_xlabel('Emotion')
    ax.set_ylabel(title)
    plt.sca(ax)
    plt.xticks(rotation=0)

plt.suptitle('Acoustic Feature Distributions per Emotion', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../logs/feature_acoustic.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. MFCC Heatmap per Emotion

MFCCs capture timbral texture. Compare the mean MFCC profiles across emotions.

In [ ]:
N_MFCC = 40
avg_mfcc = {}

for emo in EMOTIONS:
    subset = df[df['emotion'] == emo].sample(min(30, len(df[df['emotion']==emo])), random_state=1)
    mfccs = []
    for _, row in subset.iterrows():
        try:
            y, sr = librosa.load(row['audio_path'], sr=SR)
            mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
            mfccs.append(mfcc.mean(axis=1))
        except Exception:
            continue
    if mfccs:
        avg_mfcc[emo] = np.mean(mfccs, axis=0)

mfcc_matrix = np.stack([avg_mfcc[e] for e in EMOTIONS], axis=1)  # [N_MFCC, 6]

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(mfcc_matrix, ax=ax, xticklabels=[e.upper() for e in EMOTIONS],
            yticklabels=[f'MFCC {i+1}' for i in range(N_MFCC)],
            cmap='coolwarm', center=0, linewidths=0.3)
ax.set_title('Mean MFCC Coefficients per Emotion', fontsize=13, fontweight='bold')
ax.set_xlabel('Emotion')
ax.set_ylabel('MFCC Coefficient')
plt.tight_layout()
plt.savefig('../logs/feature_mfcc_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()